In [2]:
import pandas as pd
import math
import pandas as pd
import numpy as np
import statistics
from statistics import mean
import json
import pickle

In [3]:
from pyserini.index.lucene import IndexReader

# Initialize from a pre-built index:
index_reader = IndexReader.from_prebuilt_index('msmarco-v1-passage')

# Initialize from an index path:
# index_reader = IndexReader('indexes/index-robust04-20191213/')

In [4]:
index_stats = index_reader.stats()
print(index_stats)

{'total_terms': 352316036, 'documents': 8841823, 'non_empty_documents': 8841823, 'unique_terms': 2660824}


In [5]:
collection = pd.read_csv("collection.tsv" , sep='\t' , names=['docid','passage'])

In [281]:
dev_queries = pd.read_csv("Dev/queries.txt" , sep='\t' , names=['id' , 'query'])
# dev_queries

In [7]:
hard_queries = pd.read_csv("Dl_hard/topics.tsv" , sep='\t' , names=['id' , 'query'])
# hard_queries

In [119]:
Dl_2019_queries = pd.read_csv("Dl_2019/Queries.tsv" , sep='\t' , names=['id' , 'query'])
# Dl_2019_queries

In [201]:
Dl_2020_queries = pd.read_csv("Dl_2020/Queries.tsv" , sep='\t' , names=['id' , 'query'])
# Dl_2020_queries

In [282]:
bm25scores = pd.read_csv("Dev/BM25_dev.txt" , sep=' ', names=['qid','C2','docid','C4','score','C6'])
# bm25scores

In [284]:
bm25scores = bm25scores[['qid', 'docid', 'score']]

bm25scores['qid'] = bm25scores['qid'].astype(int)
bm25scores['docid'] = bm25scores['docid'].astype(int)
bm25scores['score'] = bm25scores['score'].astype(float)
# bm25scores

In [285]:
all_result = bm25scores.merge(collection, how='left', on=['docid']).fillna(0)
all_result

,qid,docid,score,passage
0,1048585,7187158,17.949499,Paula Deen and her brother Earl W. Bubba Hiers...
1,1048585,7187157,17.665600,The New York Times. U.S. | National Briefing |...
2,1048585,7187163,17.390600,Racial scandals aren't always bad for business...
3,1048585,7546327,17.034100,What happened to Paula Deen's first husband? k...
4,1048585,7187160,16.565201,Paula Deen & Brother Bubba Sued for Harassment...
...,...,...,...,...
6974593,1048565,5442567,5.047699,"Grant Gustin, who plays Barry Allen in the 201..."
6974594,1048565,7287110,5.047698,Pennywise: Who is the clown from IT? Which act...
6974595,1048565,7734168,5.047697,The actress who played the youngest daughter o...
6974596,1048565,7737804,5.047696,"Jeff Sears, who plays Prince Eric, and Michell..."


In [14]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

def clean_text(text):

#  This function takes as input a text on which several
#  NLTK algorithms will be applied in order to preprocess it

    tokens = word_tokenize(text)
    # Remove the punctuations
    tokens = [word for word in tokens if word.isalpha()]
    # Lower the tokens
    tokens = [word.lower() for word in tokens]
    # Remove stopword
    tokens = [word for word in tokens if not word in stopwords.words("english")]
    # Lemmatize
    lemma = WordNetLemmatizer()
    tokens = [lemma.lemmatize(word, pos = "v") for word in tokens]
    tokens = [lemma.lemmatize(word, pos = "n") for word in tokens]
    return tokens

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [224]:
NDCG_scores = pd.read_csv("DL_2020/NDCG.tsv" , sep='\t' , names=['qid','text','score'])
NDCG_scores = NDCG_scores[['qid', 'score']]
# print(NDCG_scores)

In [19]:
from collections import Counter
# k1=0.9, b=0.4
def bm25_similarity(query, document, k1=1.2, b=0.75):
    score = 0.0
    N = 8841823  # Total number of documents in the corpus
    avgdl = 32
    # avgdl = sum(len(doc) for doc in corpus) / N  # Average document length in the corpus
    # analyzed_query = index_reader.analyze(query)
    # print(analyzed_query)
    document = clean_text(all_doc)
    query = clean_text(query)
    x = Counter(document)
    
    for term in query:
        # analyzed = index_reader.analyze(term)
        # print(analyzed)
        
        # Skip term analysis:
        df, cf = index_reader.get_term_counts(term, analyzer=None)
        # print(f'term "{term}": df={df}, cf={cf}')
        tf = x[term]
        # print(tf)
    
        # Compute the BM25 components
        idf = math.log((N - df + 0.5) / (df + 0.5))
        term_score = (idf * tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (len(document) / avgdl)))
        # print(term_score)
    
        score += term_score

    return score

In [20]:
import math
from collections import Counter

def bm25_similarity1(query, document, k1=0.9, b=0.4):
    score = 0.0
    N = 8841823  # Total number of documents in the corpus
    avgdl = 32  # Average document length in the corpus

    analyzed_query = index_reader.analyze(query)
    # print(analyzed_query)
    analyzed_document = index_reader.analyze(document)
    # print(analyzed_document)

    for term in analyzed_query:
        
        df, cf = index_reader.get_term_counts(term, analyzer=None)
        # print(df)
        
        # Compute term frequency (tf) in the document
        tf = analyzed_document.count(term)
        # print(tf)

        # Compute the BM25 components
        idf = math.log((N - df + 0.5) / (df + 0.5))
        # print(idf)
        term_score = (idf * tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (len(analyzed_document) / avgdl)))
        score += term_score

    return score

In [541]:
# calculate Score BM25 5_1 Kmeans
results = pd.DataFrame(columns=['qid','score'])
# my_queries = dev_queries.head(1)
for i,row in hard_queries.iterrows():
    qid = row['id']
    query = row['query']
    retrieved_list = all_result.loc[all_result['qid'] == qid]
    
    res = next((item for item in clustered_data if item["qid"] == qid), None)
    lables = res["lables"]
    
    df = pd.DataFrame({'qid': retrieved_list["qid"], 'docid': retrieved_list["docid"], 'score': retrieved_list["score"],
                       'passage': retrieved_list["passage"], 'cluster': lables})

    all_doc = ""
    for g, data in df.groupby('cluster'):
        best_docs = data.head(1)
        # print(best_docs)
        for j , best_doc in best_docs.iterrows():
            all_doc += best_doc["passage"]

    BM25_score = bm25_similarity1(query, all_doc)
        
    new_data = pd.DataFrame([[ qid , BM25_score ]],columns= results.columns)
    results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('DL_hard/K-means/bm25_scores/51.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_17364\2541652878.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids
import matplotlib.pyplot as plt
from decimal import Decimal
from scipy.spatial.distance import euclidean

my_list = list()

# my_queries = dev_queries.head(1)
for i,row in dev_queries.iterrows():
  qid = row['id']
  retrieved_list = all_result.loc[all_result['qid'] == qid]
  documents = retrieved_list["passage"]

  # create vectorizer
  vectorizer = TfidfVectorizer(stop_words='english')

  # vectorizer the text documents
  vectorized_documents = vectorizer.fit_transform(documents)
  # print(vectorized_documents)
  matrix = vectorized_documents.toarray()
  # print(matrix)

  # reduce the dimensionality of the data using PCA
  pca = PCA(n_components=2)
  reduced_data = pca.fit_transform(vectorized_documents.toarray())

  num_clusters = 5

  # Cluster the data using KMedoids
  kmedoids = KMedoids(n_clusters=num_clusters,max_iter=500, random_state=42)
  kmedoids.fit(vectorized_documents)
  labels = kmedoids.labels_
  medoid_indices = kmedoids.medoid_indices_
    
  # Cluster the documents using K-Means
  # kmeans = KMeans(n_clusters=num_clusters, n_init="auto",max_iter=500, random_state=42)
  # kmeans.fit(vectorized_documents)
  # labels = kmeans.labels_  

# Loop over all clusters and find index of closest point to the cluster center and append to closest_pt_idx list.
  # closest_pt_idx = []
  # for iclust in range(kmeans.n_clusters):
  #     # get all points assigned to each cluster:
  #     cluster_pts = matrix[kmeans.labels_ == iclust]
   
  #     # get all indices of points assigned to this cluster:
  #     cluster_pts_indices = np.where(kmeans.labels_ == iclust)[0]

  #     cluster_cen = kmeans.cluster_centers_[iclust]
  #     min_idx = np.argmin([euclidean(matrix[idx], cluster_cen) for idx in cluster_pts_indices])
  #     closest_pt_idx.append(cluster_pts_indices[min_idx])

  my_obj = {"qid": qid,
            "lables": labels,
            "medoids_indices": medoid_indices
           }

  my_list.append(my_obj)
# print(my_list)

# Store data (serialize)
with open('Dev/K-medoids/five_clusters.pickle', 'wb') as handle:
    pickle.dump(my_list, handle, protocol=pickle.HIGHEST_PROTOCOL)


C:\Users\Akaberi.mozhgan\anaconda3\lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 1 is empty! self.labels_[self.medoid_indices_[1]] may not be labeled with its corresponding cluster (1).
  warnings.warn(
C:\Users\Akaberi.mozhgan\anaconda3\lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 2 is empty! self.labels_[self.medoid_indices_[2]] may not be labeled with its corresponding cluster (2).
  warnings.warn(
C:\Users\Akaberi.mozhgan\anaconda3\lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 1 is empty! self.labels_[self.medoid_indices_[1]] may not be labeled with its corresponding cluster (1).
  warnings.warn(
C:\Users\Akaberi.mozhgan\anaconda3\lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 2 is empty! self.labels_[self.medoid_indices_[2]] may not be labeled with its corresponding cluster (2).
  warnings.warn(
C:\Users\Akaberi.mozhgan\anaconda3\lib\site-packages

In [768]:
# Load data (deserialize)
with open('Dev/K-means/ten_clusters.pickle', 'rb') as handle:
    clustered_data = pickle.load(handle)

# print(type(clustered_data))    
print(len(clustered_data))

6980


In [769]:
with open('Dev/ScoreCorpusPerQueryDev', 'rb') as handle:
    ScoreCorpus = pickle.load(handle)
   
print(len(ScoreCorpus))
# print(ScoreCorpus)

6980


In [291]:
RR_scores = pd.read_csv("Dev/rr.txt" , sep='\t' , names=['RR','qid','score'])
RR_scores = RR_scores[['qid','score']]
# RR_scores

In [292]:
# calculate Primary WIG

# my_queries = dev_queries.head(2)
results = pd.DataFrame(columns=['qid','score'])
for i,row in dev_queries.iterrows():
  qid = row['id']
  query = row['query']
  tokenized_query = query.split(" ")
  query_len = 1 / math.sqrt(len(tokenized_query))

  retrieved_list = bm25scores.loc[bm25scores['qid'] == qid]
  wig_score = 0
  score_D = ScoreCorpus[f"{qid}"]
    
  for index, row1 in retrieved_list.iterrows():
      score_d = row1['score']
      wig_score += (score_d - score_D)

  wig_score = wig_score / len(retrieved_list) * query_len
  new_data = pd.DataFrame([[ qid , wig_score ]],columns= results.columns)
  results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('Dev/wig_scores_base.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_10440\3701087710.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [293]:
# calculate Primary SMV
results = pd.DataFrame(columns=['qid','score'])
for i,row in dev_queries.iterrows():
  qid = row['id']
  retrieved_list = bm25scores.loc[bm25scores['qid'] == qid]
  k = len(retrieved_list)
  score_D = ScoreCorpus[f"{qid}"]
  smv_score = 0

  mean_score = mean(retrieved_list['score'])
  for index, row1 in retrieved_list.iterrows():
      doc_score = row1['score']
      w = abs(np.log(doc_score / mean_score))
      smv_score += doc_score * w

  smv_score = smv_score / k
  smv_score = smv_score / score_D
  # print(smv_score)

  new_data = pd.DataFrame([[ qid , smv_score ]],columns= results.columns)
  results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('Dev/smv_scores_base.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_10440\1333523763.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [294]:
# calculate Primary NQC
# score_corpus = pd.read_csv("DL_hard/score_D.txt" , sep='\t' , names=['qid','score'])
# score_D = score_corpus.loc[score_corpus['qid'] == qid, 'score'].item()

results = pd.DataFrame(columns=['qid','score'])
# my_queries = dev_queries.head(1)
for i,row in dev_queries.iterrows():
  qid = row['id']
  retrieved_list = bm25scores.loc[bm25scores['qid'] == qid]
  nqc_score = 0
  score_D = ScoreCorpus[f"{qid}"]
    
  mean_score = mean(retrieved_list['score'])
  for index, row1 in retrieved_list.iterrows():
    score_d = row1['score']
    nqc_score += pow((score_d - mean_score),2)
    
  nqc_score = nqc_score / len(retrieved_list)
  nqc_score = math.sqrt(nqc_score)
    
  nqc_score = nqc_score / score_D
    
  new_data = pd.DataFrame([[ qid , nqc_score ]],columns= results.columns)
  results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('Dev/nqc_scores_base.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_10440\674853292.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [698]:
# Calculate Clustering NQC
# score_bm25 = pd.read_csv("DL_hard/K-means/bm25_scores/51.txt" , sep='\t' , names=['qid','score'])
results = pd.DataFrame(columns=['qid','score'])
# my_queries = dev_queries.head(2)
for i,row in dev_queries.iterrows():
    qid = row['id']
    retrieved_list = all_result.loc[all_result['qid'] == qid]
    
    res = next((item for item in clustered_data if item["qid"] == qid), None)
    lables = res["lables"]
    medoids_indices = res["medoids_indices"]

    nqc_score = 0
    # score_D = score_bm25.loc[score_bm25['qid'] == qid, 'score'].item()
    score_D = ScoreCorpus[f"{qid}"]
   
    df = pd.DataFrame({'qid': retrieved_list["qid"], 'docid': retrieved_list["docid"], 'score': retrieved_list["score"],'cluster': lables})

    for g, data in df.groupby('cluster'):
        medoid_doc = retrieved_list['score'].iloc[medoids_indices[g]]
        best_docs = data.head(10)
        for j,row1 in best_docs.iterrows():
            best_doc_score = row1["score"]
            nqc_score += pow((best_doc_score - medoid_doc),2)
            
    k = len(medoids_indices) * len(best_docs)
#     # print(k)
    nqc_score = nqc_score / k

    nqc_score = math.sqrt(nqc_score)
    nqc_score = nqc_score / score_D
        
    new_data = pd.DataFrame([[ qid , nqc_score ]],columns= results.columns)
    results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('Dev/K-means/nqc_scores/510.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_10440\3126915792.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


KeyboardInterrupt: 

In [637]:
# Calculate Clustering WIG
results = pd.DataFrame(columns=['qid','score'])
# my_queries = hard_queries.head(1)
for i,row in dev_queries.iterrows():
    qid = row['id']
    query = row['query']
    tokenized_query = query.split(" ")
    query_len = 1 / math.sqrt(len(tokenized_query))
    
    retrieved_list = all_result.loc[all_result['qid'] == qid]
    wig_score = 0
    score_D = ScoreCorpus[f"{qid}"]
    
    res = next((item for item in clustered_data if item["qid"] == qid), None)
    lables = res["lables"]
    medoids_indices = res["medoids_indices"]
   
    df = pd.DataFrame({'qid': retrieved_list["qid"], 'docid': retrieved_list["docid"], 'score': retrieved_list["score"],'cluster': lables})
   
    for g, data in df.groupby('cluster'):
        best_docs = data.head(20)
        for j,row1 in best_docs.iterrows():
            best_doc_score = row1["score"]
            wig_score += (best_doc_score - score_D)

    k = len(medoids_indices) * len(best_docs)
    # print(k)
    wig_score = wig_score / k * query_len
    # print(wig_score)
        
    new_data = pd.DataFrame([[ qid , wig_score ]],columns= results.columns)
    results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('Dev/K-means/wig_scores/1020.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_10440\1187256031.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [782]:
# Calculate Clustering SMV
results = pd.DataFrame(columns=['qid','score'])
# my_queries = hard_queries.head(1)
for i,row in dev_queries.iterrows():
    qid = row['id']
    retrieved_list = all_result.loc[all_result['qid'] == qid]
    
    res = next((item for item in clustered_data if item["qid"] == qid), None)
    lables = res["lables"]
    medoids_indices = res["medoids_indices"]

    smv_score = 0
    score_D = ScoreCorpus[f"{qid}"]
    df = pd.DataFrame({'qid': retrieved_list["qid"], 'docid': retrieved_list["docid"], 'score': retrieved_list["score"],'cluster': lables})
   
    for g, data in df.groupby('cluster'):
        medoid_doc = retrieved_list['score'].iloc[medoids_indices[g]]
        best_docs = data.head(5)
        for j,row1 in best_docs.iterrows():
            best_doc_score = row1["score"]
            w = abs(np.log(best_doc_score / medoid_doc))
            smv_score += best_doc_score * w

    k = len(medoids_indices) * len(best_docs)
    # print(k)
    smv_score = smv_score / k
    smv_score = smv_score / score_D
        
    new_data = pd.DataFrame([[ qid , smv_score ]],columns= results.columns)
    results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('Dev/K-means/smv_scores/105.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_10440\614164384.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [798]:
# nqc_scores = pd.read_csv("Dev/wig_scores_base.txt" , sep='\t' , names=['qid','score'])
nqc_scores = pd.read_csv("Dev/K-means/nqc_scores/d51.txt" , sep='\t' , names=['qid','score'])
# print(nqc_scores)

x = RR_scores
# x = NDCG_scores
y = nqc_scores

sorted_x = x.sort_values(by=['qid'])
# sorted_x

sorted_y = y.sort_values(by=['qid'])
# sorted_y

In [799]:
my_data = sorted_x.merge(sorted_y, how='left', on=['qid']).fillna(0)
my_data = my_data[['score_x' , 'score_y']]
# my_data
correlation = my_data['score_x'].corr(my_data['score_y'])
print(correlation)

0.18505559000975763


In [800]:
corr = my_data.corr(method = 'pearson')
corr

,score_x,score_y
score_x,1.000000,0.185056
score_y,0.185056,1.000000


In [801]:
corr = my_data.corr(method = 'spearman')
corr

,score_x,score_y
score_x,1.00000,0.22788
score_y,0.22788,1.00000


In [802]:
corr = my_data.corr(method = 'kendall')
corr

,score_x,score_y
score_x,1.000000,0.170976
score_y,0.170976,1.000000
